# LiveOps Recommendations and Dashboard Marts

This notebook builds the final PlayerPulse LiveOps recommendation and dashboard summary layer from existing processed outputs only. It does not rebuild raw parsing, staging, daily metrics, cohort retention, segmentation, or forecasting.

Inputs:
- `data/processed/agg_daily_metrics.parquet`
- `data/processed/agg_cohort_retention.parquet`
- `data/processed/agg_segment_monthly.parquet`
- `data/processed/mart_player_segments.parquet`
- `data/processed/mart_forecast_daily.parquet`
- `data/processed/mart_forecast_backtest.parquet`
- `data/processed/mart_trend_alerts.parquet`

Outputs:
- `data/processed/mart_liveops_recommendations.parquet`
- `data/processed/dashboard_kpi_summary.parquet`
- `data/processed/dashboard_retention_summary.parquet`
- `data/processed/dashboard_segment_summary.parquet`
- `data/processed/dashboard_forecast_summary.parquet`
- `data/outputs/liveops_recommendation_summary.csv`

## Approach

The final layer turns analytical signals into operational artifacts:
- Trend alerts become ranked LiveOps recommendations.
- Daily, retention, segment, and forecast marts become compact dashboard-ready summary tables.
- Validation checks make sure recommendations are unique, ranked, actionable, and dashboard tables are populated.

`mart_trend_alerts` is the main recommendation source because it already combines activity, retention, and segment movement signals.

In [ ]:
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

processed_dir = project_root / 'data' / 'processed'
outputs_dir = project_root / 'data' / 'outputs'
sql_path = project_root / 'sql' / '06_liveops_recommendations_dashboard_marts.sql'

input_paths = {
    'agg_daily_metrics': processed_dir / 'agg_daily_metrics.parquet',
    'agg_cohort_retention': processed_dir / 'agg_cohort_retention.parquet',
    'agg_segment_monthly': processed_dir / 'agg_segment_monthly.parquet',
    'mart_player_segments': processed_dir / 'mart_player_segments.parquet',
    'mart_forecast_daily': processed_dir / 'mart_forecast_daily.parquet',
    'mart_forecast_backtest': processed_dir / 'mart_forecast_backtest.parquet',
    'mart_trend_alerts': processed_dir / 'mart_trend_alerts.parquet',
}

output_paths = {
    'mart_liveops_recommendations': processed_dir / 'mart_liveops_recommendations.parquet',
    'dashboard_kpi_summary': processed_dir / 'dashboard_kpi_summary.parquet',
    'dashboard_retention_summary': processed_dir / 'dashboard_retention_summary.parquet',
    'dashboard_segment_summary': processed_dir / 'dashboard_segment_summary.parquet',
    'dashboard_forecast_summary': processed_dir / 'dashboard_forecast_summary.parquet',
}
validation_summary_path = outputs_dir / 'liveops_recommendation_summary.csv'

missing = [str(path.relative_to(project_root)) for path in list(input_paths.values()) + [sql_path] if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

print('Project root:', project_root)
print('Inputs:', [str(path.relative_to(project_root)) for path in input_paths.values()])

## Build DuckDB Views

Each processed parquet is registered as a DuckDB view. The SQL file then creates recommendation and dashboard summary views from those inputs.

In [ ]:
con = duckdb.connect()
for view_name, path in input_paths.items():
    con.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_parquet('{path.as_posix()}')")

con.execute(sql_path.read_text())

source_counts = []
for view_name in input_paths:
    row_count = con.execute(f'SELECT COUNT(*) FROM {view_name}').fetchone()[0]
    source_counts.append({'source': view_name, 'row_count': row_count})

pd.DataFrame(source_counts)

## Export Dashboard Marts

DuckDB writes the parquet outputs directly so the notebook does not depend on a pandas parquet engine.

In [ ]:
for view_name, output_path in output_paths.items():
    con.execute(f"COPY (SELECT * FROM {view_name}) TO '{output_path.as_posix()}' (FORMAT PARQUET)")
    print('Wrote:', output_path.relative_to(project_root))

## Validation

The final layer is checked for duplicate recommendation IDs, unique priority ranking, missing actions, empty dashboard tables, and addressable segment share integrity.

In [ ]:
recommendations = con.execute('SELECT * FROM mart_liveops_recommendations ORDER BY priority_rank').df()
kpi_summary = con.execute('SELECT * FROM dashboard_kpi_summary').df()
retention_summary = con.execute('SELECT * FROM dashboard_retention_summary ORDER BY cohort_date').df()
segment_summary = con.execute('SELECT * FROM dashboard_segment_summary ORDER BY segment_size DESC').df()
forecast_summary = con.execute('SELECT * FROM dashboard_forecast_summary ORDER BY forecast_date').df()

validation = {}
validation['recommendation_rows'] = len(recommendations)
validation['duplicate_recommendation_id_rows'] = int(recommendations['recommendation_id'].duplicated().sum()) if not recommendations.empty else 0
validation['duplicate_priority_rank_rows'] = int(recommendations['priority_rank'].duplicated().sum()) if not recommendations.empty else 0
validation['null_recommended_action_rows'] = int(recommendations['recommended_action'].isna().sum()) if not recommendations.empty else 0
validation['dashboard_kpi_summary_rows'] = len(kpi_summary)
validation['dashboard_retention_summary_rows'] = len(retention_summary)
validation['dashboard_segment_summary_rows'] = len(segment_summary)
validation['dashboard_forecast_summary_rows'] = len(forecast_summary)

addressable_share_sum = segment_summary['addressable_segment_share'].dropna().sum() if not segment_summary.empty else 0
validation['addressable_segment_share_sum'] = float(addressable_share_sum)
validation['addressable_segment_share_not_close_to_1'] = int(abs(addressable_share_sum - 1.0) > 0.001)

warnings_list = []
if validation['duplicate_recommendation_id_rows']:
    warnings_list.append('duplicate_recommendation_id_rows')
if validation['duplicate_priority_rank_rows']:
    warnings_list.append('duplicate_priority_rank_rows')
if validation['null_recommended_action_rows']:
    warnings_list.append('null_recommended_action_rows')
for table_name in ['dashboard_kpi_summary', 'dashboard_retention_summary', 'dashboard_segment_summary', 'dashboard_forecast_summary']:
    if validation[f'{table_name}_rows'] == 0:
        warnings_list.append(f'{table_name}_empty')
if validation['addressable_segment_share_not_close_to_1']:
    warnings_list.append('addressable_segment_share_not_close_to_1')

if warnings_list:
    print('Warnings:', warnings_list)
else:
    print('Validation passed with no warnings')

pd.DataFrame([validation])

## Recommendation Summary CSV

This CSV is intentionally compact: it stores headline row counts, warnings, KPI values, latest segment markers, forecast range, and the recommendation list for handoff.

In [ ]:
latest_kpi = kpi_summary.iloc[0].to_dict() if not kpi_summary.empty else {}
latest_segment_snapshot = segment_summary['snapshot_date'].iloc[0] if not segment_summary.empty else None
largest_segment = segment_summary['largest_segment'].iloc[0] if not segment_summary.empty else None
fastest_growing_segment = segment_summary['fastest_growing_segment'].iloc[0] if not segment_summary.empty else None
fastest_shrinking_segment = segment_summary['fastest_shrinking_segment'].iloc[0] if not segment_summary.empty else None
forecast_start = forecast_summary['final_forecast_start_date'].iloc[0] if not forecast_summary.empty else None
forecast_end = forecast_summary['final_forecast_end_date'].iloc[0] if not forecast_summary.empty else None
champion_model = forecast_summary['champion_model'].iloc[0] if not forecast_summary.empty else None
champion_mae = forecast_summary['champion_mae'].iloc[0] if not forecast_summary.empty else None
champion_smape = forecast_summary['champion_smape'].iloc[0] if not forecast_summary.empty else None

summary_rows = [
    {'section': 'files', 'metric_name': name, 'metric_value': str(path.relative_to(project_root))}
    for name, path in output_paths.items()
]
summary_rows.append({'section': 'files', 'metric_name': 'liveops_recommendation_summary', 'metric_value': str(validation_summary_path.relative_to(project_root))})
summary_rows.extend([
    {'section': 'validation', 'metric_name': key, 'metric_value': value}
    for key, value in validation.items()
])
summary_rows.append({'section': 'validation', 'metric_name': 'warnings', 'metric_value': '; '.join(warnings_list) if warnings_list else 'none'})

for key, value in latest_kpi.items():
    summary_rows.append({'section': 'latest_kpi_summary', 'metric_name': key, 'metric_value': value})

summary_rows.extend([
    {'section': 'latest_segment_summary', 'metric_name': 'latest_segment_snapshot', 'metric_value': latest_segment_snapshot},
    {'section': 'latest_segment_summary', 'metric_name': 'largest_segment', 'metric_value': largest_segment},
    {'section': 'latest_segment_summary', 'metric_name': 'fastest_growing_segment', 'metric_value': fastest_growing_segment},
    {'section': 'latest_segment_summary', 'metric_name': 'fastest_shrinking_segment', 'metric_value': fastest_shrinking_segment},
    {'section': 'forecast_summary', 'metric_name': 'champion_model', 'metric_value': champion_model},
    {'section': 'forecast_summary', 'metric_name': 'champion_mae', 'metric_value': champion_mae},
    {'section': 'forecast_summary', 'metric_name': 'champion_smape', 'metric_value': champion_smape},
    {'section': 'forecast_summary', 'metric_name': 'forecast_date_range', 'metric_value': f'{forecast_start} to {forecast_end}'},
])

for _, row in recommendations.iterrows():
    summary_rows.append({
        'section': 'recommendations',
        'metric_name': f"priority_{int(row['priority_rank'])}_{row['source_alert_type']}",
        'metric_value': row['recommended_action'],
    })

validation_summary_df = pd.DataFrame(summary_rows)
validation_summary_df.to_csv(validation_summary_path, index=False)
validation_summary_df.head(30)

## Final Report Values

The final cell prints the exact values needed for the project handoff.

In [ ]:
top_5_recommendations = recommendations[[
    'priority_rank', 'severity', 'target_segment', 'issue_detected', 'recommended_action', 'pct_change'
]].head(5)

latest_segment_summary = segment_summary[[
    'lifecycle_segment', 'segment_size', 'segment_share', 'addressable_segment_share', 'segment_size_change_3mo'
]].head(10)

forecast_report = forecast_summary[[
    'champion_model', 'champion_mae', 'champion_rmse', 'champion_smape',
    'final_forecast_start_date', 'final_forecast_end_date', 'average_forecast_dau_30d'
]].head(1)

print('Files created:')
for path in list(output_paths.values()) + [validation_summary_path]:
    print('-', path.relative_to(project_root))

print('\nNumber of recommendations:', len(recommendations))
print('\nTop 5 recommendations:')
display(top_5_recommendations)
print('\nLatest KPI summary:')
display(kpi_summary)
print('\nLatest segment summary:')
display(latest_segment_summary)
print('\nForecast summary:')
display(forecast_report)
print('\nAny warnings:', warnings_list or ['none'])